Description : Import de toutes les bibliothèques nécessaires, fixation de la graine aléatoire pour la reproductibilité, et création du dossier data/.

In [8]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)

print("=== Prêt ===")

=== Prêt ===


Description : Définition de toutes les valeurs de référence utilisées lors de la génération : tranches d'âge, régions, abonnements, mobilités, intérêts, catégories, prestations par catégorie, niveaux et statuts.


In [9]:
tranches_age = ['60-69', '70-79', '80+']

regions = ['Île-de-France', 'PACA', 'Auvergne-Rhône-Alpes', 'Occitanie', 'Bretagne']

abonnements = ['Basique', 'Premium']

mobilites = ['Faible', 'Moyenne', 'Élevée']

tous_interets = ['santé', 'loisirs', 'administratif', 'accompagnement']

categories = ['Santé', 'Loisirs', 'Accompagnement', 'Administratif']

prestations_par_categorie = {
    'Santé':          ['Suivi médical', 'Atelier bien-être', 'Séance kiné', 'Nutrition seniors'],
    'Loisirs':        ['Atelier mémoire', 'Sortie culturelle', 'Atelier créatif', 'Jeux de société'],
    'Accompagnement': ['Aide courses', 'Visite à domicile', 'Soutien psychologique', 'Transport'],
    'Administratif':  ['Aide démarches', 'Gestion courrier', 'Accès numérique', 'Fiscalité']
}

niveaux = ['Facile', 'Moyen', 'Avancé']

statuts = ['Confirmé', 'Annulé', 'Terminé']

print("=== Valeurs de référence définies ===")
print(f"Tranches d'âge     : {tranches_age}")
print(f"Régions            : {regions}")
print(f"Catégories cibles  : {categories}")

=== Valeurs de référence définies ===
Tranches d'âge     : ['60-69', '70-79', '80+']
Régions            : ['Île-de-France', 'PACA', 'Auvergne-Rhône-Alpes', 'Occitanie', 'Bretagne']
Catégories cibles  : ['Santé', 'Loisirs', 'Accompagnement', 'Administratif']


Description : Génération des 500 lignes avec des règles simples : l'âge et la mobilité influencent la catégorie probable, l'ancienneté augmente les chances de réinscription.

In [10]:
lignes = []

for i in range(500):
    client_id = f"C{str(i+1).zfill(3)}"
    tranche_age = np.random.choice(['60-69', '70-79', '80+'], p=[0.30, 0.45, 0.25])
    region = np.random.choice(['Île-de-France', 'PACA', 'Auvergne-Rhône-Alpes', 'Occitanie', 'Bretagne'])
    abonnement_type = np.random.choice(['Basique', 'Premium'], p=[0.70, 0.30])
    anciennete_mois = np.random.randint(0, 61)

    if tranche_age == '80+':
        mobilite = np.random.choice(['Faible', 'Moyenne', 'Élevée'], p=[0.55, 0.35, 0.10])
    elif tranche_age == '70-79':
        mobilite = np.random.choice(['Faible', 'Moyenne', 'Élevée'], p=[0.35, 0.45, 0.20])
    else:
        mobilite = np.random.choice(['Faible', 'Moyenne', 'Élevée'], p=[0.20, 0.45, 0.35])

    nb_interets = np.random.randint(1, 4)
    interets = ','.join(np.random.choice(['santé', 'loisirs', 'administratif', 'accompagnement'], size=nb_interets, replace=False))

    if tranche_age == '80+' or mobilite == 'Faible':
        proba_cat = [0.35, 0.15, 0.40, 0.10]
    elif tranche_age == '60-69' and mobilite == 'Élevée':
        proba_cat = [0.15, 0.45, 0.10, 0.30]
    else:
        proba_cat = [0.25, 0.30, 0.25, 0.20]

    categories = ['Santé', 'Loisirs', 'Accompagnement', 'Administratif']
    prestations = {
        'Santé': ['Suivi médical', 'Atelier bien-être', 'Séance kiné'],
        'Loisirs': ['Atelier mémoire', 'Sortie culturelle', 'Atelier créatif'],
        'Accompagnement': ['Aide courses', 'Visite à domicile', 'Transport'],
        'Administratif': ['Aide démarches', 'Gestion courrier', 'Accès numérique']
    }

    categorie = np.random.choice(categories, p=proba_cat)
    prestation = np.random.choice(prestations[categorie])
    prix = round(np.random.uniform(20, 80), 2)
    duree_min = np.random.choice([30, 60, 90])
    niveau = np.random.choice(['Facile', 'Moyen', 'Avancé'], p=[0.50, 0.35, 0.15])
    mois = f"2025-{str(np.random.randint(1, 13)).zfill(2)}"
    inscription = 1
    reinscription = int(np.random.random() < min(0.3 + anciennete_mois * 0.01, 0.9))
    statut = np.random.choice(['Confirmé', 'Annulé', 'Terminé'], p=[0.20, 0.08, 0.72])

    lignes.append({
        'client_id': client_id, 'tranche_age': tranche_age, 'region': region,
        'abonnement_type': abonnement_type, 'anciennete_mois': anciennete_mois,
        'mobilite': mobilite, 'interets': interets, 'categorie': categorie,
        'prestation': prestation, 'prix': prix, 'duree_min': duree_min,
        'niveau': niveau, 'mois': mois, 'inscription': inscription,
        'reinscription': reinscription, 'statut': statut
    })

df = pd.DataFrame(lignes)

print("=== Dataset généré ===")
print(f"Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df.head())

=== Dataset généré ===
Dimensions : 500 lignes, 16 colonnes
  client_id tranche_age                region abonnement_type  \
0      C001       70-79              Bretagne         Basique   
1      C002       60-69             Occitanie         Basique   
2      C003       70-79              Bretagne         Basique   
3      C004       70-79  Auvergne-Rhône-Alpes         Premium   
4      C005       70-79             Occitanie         Basique   

   anciennete_mois mobilite                            interets  \
0                7  Moyenne  accompagnement,santé,administratif   
1               11  Moyenne                             loisirs   
2               20  Moyenne               administratif,loisirs   
3               35   Faible                 santé,administratif   
4               39   Élevée                      accompagnement   

        categorie         prestation   prix  duree_min  niveau     mois  \
0           Santé      Suivi médical  56.07         90  Facile  2025-02

Description : Vérification de la qualité : valeurs manquantes, répartition des catégories, taux de réinscription et d'annulation.

In [11]:
print("=== Qualité ===")
print(f"Valeurs manquantes : {df.isnull().sum().sum()}")
print(f"\nRépartition catégories :\n{df['categorie'].value_counts()}")
print(f"\nTaux de réinscription : {df['reinscription'].mean():.2%}")
print(f"Taux d'annulation    : {(df['statut'] == 'Annulé').mean():.2%}")

=== Qualité ===
Valeurs manquantes : 0

Répartition catégories :
categorie
Accompagnement    156
Santé             149
Loisirs           109
Administratif      86
Name: count, dtype: int64

Taux de réinscription : 61.00%
Taux d'annulation    : 10.40%


Description : Sauvegarde du dataset en CSV dans le dossier data/.

In [12]:
df.to_csv('data/dataset.csv', index=False)

print("=== Sauvegardé ===")
print("data/dataset.csv")

=== Sauvegardé ===
data/dataset.csv
